# LangGraph: 그래프 기반 에이전트 워크플로우

이번 노트북에서는 **LangGraph**의 핵심 개념을 학습합니다. LangGraph는 **상태 그래프(State Graph)** 기반으로 에이전트 워크플로우를 구성하는 프레임워크입니다. CrewAI가 선언적(YAML)이었다면, LangGraph는 **명시적(Python 함수 + 그래프)**입니다.

## 개요

| 주제 | 내용 |
|------|------|
| LangGraph 소개 | 상태 그래프 기반 에이전트 프레임워크 |
| 5단계 빌드 | State → Graph Builder → Node → Edge → Compile |
| State와 Reducer | Annotated 타입 힌트와 add_messages |
| 그래프 시각화 | Mermaid 기반 그래프 렌더링 |
| LLM 없는 그래프 | 순수 Python 함수로 그래프 구성 |
| LLM 챗봇 | ChatOpenAI를 노드로 연결한 대화형 그래프 |

## 학습 목표

1. LangGraph의 핵심 개념(State, Node, Edge, Graph)을 이해하기
2. 5단계 빌드 패턴으로 그래프를 구성하고 실행하기
3. `Annotated`와 `add_messages` reducer의 역할 파악하기
4. LLM 없이 순수 Python 함수로 그래프를 구성해보기
5. ChatOpenAI를 노드로 연결하여 대화형 챗봇 만들기

---

## CrewAI와의 비교

| 특성 | CrewAI | LangGraph |
|------|--------|----------|
| **핵심 철학** | 역할 기반 협업 (선언적) | 그래프 기반 워크플로우 (명시적) |
| **구성 단위** | Agent, Task, Crew | State, Node, Edge, Graph |
| **실행 흐름** | Process가 자동 조율 | Edge로 직접 연결 |
| **노드** | 에이전트 (role, goal) | **아무 Python 함수** |
| **상태 관리** | 프레임워크가 암묵적 관리 | State 객체로 명시적 관리 |
| **유연성** | 중간 (구조화된 패턴) | 높음 (자유로운 그래프 설계) |
| **설정** | YAML + 데코레이터 | 순수 Python 코드 |

---

## 1. LangGraph 핵심 개념

LangGraph는 **상태 머신(State Machine)**의 개념을 에이전트 워크플로우에 적용합니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    LangGraph 핵심 구성 요소                        │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────┐   그래프 전체에서 공유되는 데이터                      │
│  │  State   │   BaseModel 또는 TypedDict로 정의                    │
│  └─────────┘   Reducer가 상태 업데이트 방식을 결정                  │
│                                                                     │
│  ┌─────────┐   상태를 받아서 처리하고 새 상태를 반환                │
│  │  Node    │   아무 Python 함수가 노드가 될 수 있음               │
│  └─────────┘   LLM 호출, API 호출, 계산 등 무엇이든 가능          │
│                                                                     │
│  ┌─────────┐   노드 간 연결 (실행 순서)                            │
│  │  Edge    │   일반 Edge: A → B (항상 이동)                       │
│  └─────────┘   조건부 Edge: 조건에 따라 분기                       │
│                                                                     │
│  ┌─────────┐   노드 + 엣지의 조합                                  │
│  │  Graph   │   compile() 후 invoke()로 실행                       │
│  └─────────┘   시각화(Mermaid) 가능                                │
│                                                                     │
│  ┌─────────┐   상태 업데이트 시 호출되는 함수                      │
│  │ Reducer  │   add_messages: 메시지 리스트에 새 메시지 추가       │
│  └─────────┘   Annotated[list, add_messages]로 지정                │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### LangGraph 그래프의 5단계 빌드 패턴

```
Step 1.  State 정의         ← 그래프에서 공유할 데이터 구조
  │
  ▼
Step 2.  StateGraph 생성    ← State를 기반으로 Graph Builder 생성
  │
  ▼
Step 3.  Node 추가          ← Python 함수를 노드로 등록
  │
  ▼
Step 4.  Edge 연결          ← START → 노드 → END 연결
  │
  ▼
Step 5.  Compile & Run      ← graph.compile() → graph.invoke(state)
```

---

## 2. 환경 설정

In [28]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
import random
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

API key found.


---

## 3. Annotated와 Reducer 이해하기

### 문제 상황부터 이해하기

LangGraph에서 노드(함수)는 **기존 상태를 받아서 새 상태를 반환**합니다. 이때 질문이 생깁니다:

> 노드가 새 메시지를 반환하면, 기존 메시지는 어떻게 되지?

```
기존 상태:
  messages = ["안녕하세요"]      ← 사용자가 보낸 메시지

노드가 반환:
  messages = ["반가워요!"]       ← LLM이 생성한 응답

결과는?
  (A) messages = ["반가워요!"]                     ← 기존꺼 사라짐 (덮어쓰기)
  (B) messages = ["안녕하세요", "반가워요!"]        ← 기존꺼 유지 + 새로 추가
```

챗봇이라면 당연히 **(B) 대화 내역이 누적**되어야 합니다. 하지만 Python의 기본 동작은 **(A) 덮어쓰기**입니다.

이 문제를 해결하는 것이 **Reducer**입니다.

### Reducer = "합치는 규칙"

Reducer는 **기존 값과 새 값을 어떻게 합칠지** 정하는 함수입니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│                                                                     │
│  Reducer 없음 (기본 동작):   새 값이 기존 값을 덮어쓴다            │
│                                                                     │
│    기존: ["안녕하세요"]                                              │
│    새로: ["반가워요!"]                                               │
│    결과: ["반가워요!"]              ← 대화 기록 사라짐!             │
│                                                                     │
│  ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─   │
│                                                                     │
│  add_messages Reducer:       새 값을 기존 값 뒤에 이어붙인다       │
│                                                                     │
│    기존: ["안녕하세요"]                                              │
│    새로: ["반가워요!"]                                               │
│    결과: ["안녕하세요", "반가워요!"]  ← 대화 기록 유지!             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

비유하자면:
- **Reducer 없음** = 화이트보드를 지우고 새로 씀
- **add_messages** = 화이트보드에 계속 이어서 씀

### 어떻게 지정하나? → Annotated

Python의 `Annotated`는 타입 힌트에 **추가 정보를 붙이는** 기능입니다. LangGraph는 이 자리에 reducer 함수를 넣습니다:

```python
class State(BaseModel):
    messages: Annotated[list, add_messages]
    #         ────────  ────  ────────────
    #         타입은 list    이 필드의 reducer는 add_messages
```

이렇게 선언하면 LangGraph가 자동으로:
1. 노드가 `messages`를 반환할 때
2. 기존 `messages`와 새 `messages`를 `add_messages` 함수로 합쳐줍니다

### 실제 동작 예시 (3턴 대화)

```
[턴 1] 사용자 입력 → 노드 실행
  invoke 전:  messages = [user: "안녕"]
  노드 반환:  messages = [assistant: "반가워요!"]
  reducer 후: messages = [user: "안녕", assistant: "반가워요!"]

[턴 2] 사용자 입력 → 노드 실행
  invoke 전:  messages = [user: "안녕", assistant: "반가워요!", user: "날씨 어때?"]
  노드 반환:  messages = [assistant: "오늘 맑아요!"]
  reducer 후: messages = [user: "안녕", assistant: "반가워요!",
                          user: "날씨 어때?", assistant: "오늘 맑아요!"]
```

**대화가 계속 쌓이니까** LLM이 이전 대화를 기억할 수 있는 것입니다.

### add_messages는 기본 제공 함수

`add_messages`는 LangGraph가 **내장으로 제공**하는 reducer입니다:

```python
from langgraph.graph.message import add_messages   # ← LangGraph 내장
```

가장 흔한 패턴(메시지 누적)을 위해 미리 만들어둔 것입니다. 하지만 reducer 자체는 특별한 것이 아닙니다 — `(기존값, 새값) → 합친값` 시그니처를 가진 **아무 Python 함수**면 reducer가 될 수 있습니다:

```python
# add_messages가 내부적으로 하는 일 (단순화)
def add_messages(existing: list, new: list) -> list:
    return existing + new

# 커스텀 reducer 예시: 최근 10개 메시지만 유지
def keep_last_10(existing: list, new: list) -> list:
    return (existing + new)[-10:]

# 커스텀 reducer 적용
class State(BaseModel):
    messages: Annotated[list, keep_last_10]   # ← 내가 만든 reducer 사용
```

| Reducer | 동작 | 용도 |
|---------|------|------|
| `add_messages` (내장) | 기존 + 새 메시지 이어붙임 | 일반적인 대화 누적 |
| 커스텀 `keep_last_10` | 최근 10개만 유지 | 토큰 절약, 긴 대화 |
| Reducer 없음 | 덮어쓰기 | 최신 값만 필요한 경우 |

In [29]:
# Annotated 동작 확인

def shout(text: Annotated[str, "something to be shouted"]) -> str:
    """Annotated는 실행에 영향을 주지 않고, 메타데이터만 추가합니다."""
    print(text.upper())
    return text.upper()

shout("hello")

HELLO


'HELLO'

---

## 4. 첫 번째 그래프 — LLM 없이 순수 Python 함수

LangGraph의 핵심 포인트: **노드는 아무 Python 함수**가 될 수 있습니다. LLM이 필수가 아닙니다!

5단계 빌드 패턴을 따라 LLM 없이 간단한 그래프를 만들어봅니다.

In [7]:
# 랜덤 문장 생성용 상수

nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas",
         "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody",
              "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [30]:
# ── Step 1: State 정의 ──
# messages 필드에 add_messages reducer를 지정하여 메시지가 누적되게 합니다

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [9]:
# ── Step 2: Graph Builder 생성 ──

graph_builder = StateGraph(State)

In [31]:
# ── Step 3: Node 추가 ──
# 노드는 old_state를 받아서 new_state를 반환하는 Python 함수입니다
# LLM 없이 랜덤 문장을 생성하는 노드

def our_first_node(old_state: State) -> State:
    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]
    new_state = State(messages=messages)
    return new_state

graph_builder.add_node("first_node", our_first_node)

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


In [11]:
# ── Step 4: Edge 연결 ──
# START → first_node → END

graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

In [32]:
# ── Step 5: Compile & 시각화 ──

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

### 실행해봅시다!

In [33]:
# Gradio UI로 실행

def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    state = State(messages=[message])
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### 왜 LLM 없는 그래프를 먼저 만들었을까?

무엇을 입력해도 랜덤 문장이 돌아옵니다. LLM이 아니라 Python 함수(`our_first_node`)가 응답하기 때문입니다.

이것이 바로 **LangGraph의 핵심 설계 철학**입니다:

> **노드 = 아무 Python 함수**. LLM은 그 중 하나의 선택지일 뿐!

```
┌─────────────────────────────────────────────────────────────────────┐
│                  노드가 될 수 있는 것들                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ✦ LLM 호출        →  ChatOpenAI, Anthropic, Ollama ...           │
│  ✦ API 호출        →  REST API, DB 쿼리, 웹 크롤링               │
│  ✦ 데이터 처리     →  파싱, 변환, 필터링, 집계                    │
│  ✦ 비즈니스 로직   →  유효성 검사, 규칙 엔진, 조건 판단           │
│  ✦ 도구 실행       →  파일 읽기/쓰기, 계산, 코드 실행             │
│                                                                     │
│  → 이 모든 것을 자유롭게 조합할 수 있습니다                       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### CrewAI와의 차이

CrewAI에서는 노드가 항상 **에이전트(LLM)**입니다. 반면 LangGraph는:

| | CrewAI | LangGraph |
|---|---|---|
| **노드** | 에이전트 (LLM 필수) | 아무 Python 함수 |
| **비LLM 로직** | Tool로만 가능 | 노드 자체가 될 수 있음 |
| **조합 자유도** | 에이전트 중심 | 함수 중심 (LLM은 옵션) |

이 유연성 덕분에 LangGraph로는 "LLM 호출 → 결과 검증 → 조건 분기 → 재시도" 같은 **복잡한 워크플로우**를 자연스럽게 구성할 수 있습니다.

다음 섹션에서는 노드에 LLM을 연결하여 진짜 대화가 가능한 챗봇을 만들어봅니다.

---

## 5. 두 번째 그래프 — LLM 챗봇

이제 노드에 LLM을 연결하여 진짜 대화가 가능한 챗봇을 만듭니다. 동일한 5단계 패턴을 따릅니다.

```
┌──────────────────────────────────────────────────────────┐
│             LLM 챗봇 그래프                              │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  State(messages=[user_message])                          │
│       │                                                  │
│       ▼                                                  │
│  ┌─────────────┐                                        │
│  │   chatbot    │  llm.invoke(old_state.messages)        │
│  │   (Node)     │  ChatOpenAI(gpt-4o-mini)              │
│  └─────────────┘                                        │
│       │                                                  │
│       ▼                                                  │
│  State(messages=[user_msg, assistant_msg])               │
│       │        ← add_messages reducer가 누적             │
│       ▼                                                  │
│     END                                                  │
│                                                          │
└──────────────────────────────────────────────────────────┘
```

In [22]:
# ── Step 1: State 정의 ──

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [23]:
# ── Step 2: Graph Builder 생성 ──

graph_builder = StateGraph(State)

In [24]:
# ── Step 3: Node 추가 ──
# 이번에는 ChatOpenAI를 호출하는 노드입니다

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    new_state = State(messages=[response])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)

In [25]:
# ── Step 4: Edge 연결 ──

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [26]:
# ── Step 5: Compile & 시각화 ──

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
# Gradio UI로 LLM 챗봇 실행

def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='690f0690-4e1c-4299-a2b7-c8df34f42fd4'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a1ddba3226', 'id': 'chatcmpl-DI65rqOyBEhLMgc3tUzDNxhUwmCFM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cdb4b-b63f-7cb0-9b5f-7439e315d1c9-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}
{'messag

---

## 정리

### LangGraph 5단계 빌드 패턴

```
┌─────────────────────────────────────────────────────────────────────┐
│                 LangGraph 핵심 요약                                 │
├──────────────────────────────┬──────────────────────────────────────┤
│     단계                     │     코드                             │
├──────────────────────────────┼──────────────────────────────────────┤
│  1. State 정의               │  class State(BaseModel):            │
│                              │    messages: Annotated[list,         │
│                              │              add_messages]           │
│  2. Graph Builder            │  graph_builder = StateGraph(State)  │
│  3. Node 추가                │  graph_builder.add_node(name, fn)   │
│  4. Edge 연결                │  graph_builder.add_edge(A, B)       │
│  5. Compile & Run            │  graph = graph_builder.compile()    │
│                              │  graph.invoke(state)                │
└──────────────────────────────┴──────────────────────────────────────┘
```

### 핵심 포인트

- **노드는 아무 Python 함수** — LLM이 없어도 됩니다
- **State + Reducer** — `Annotated[list, add_messages]`로 메시지 누적 관리
- **명시적 연결** — Edge로 노드 간 흐름을 직접 설계
- **시각화** — `graph.get_graph().draw_mermaid_png()`로 그래프 구조 확인

### 다음 단계

- **조건부 Edge** — 상태에 따라 다른 노드로 분기
- **Tool Use** — 에이전트가 도구를 호출하는 패턴
- **Human-in-the-loop** — 사람이 중간에 개입하는 워크플로우
- **Checkpointing** — 상태 저장 및 복구